In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

In [ ]:
import hmt_v3 as hmt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
me3_raw_df = pd.read_csv("test_data/k27_k27_thaw009_me3.csv")
ac_raw_df = pd.read_csv("test_data/k27_k27_thaw009_ac.csv")

me3_filtered_df = hmt.preprocess.filter_axial(me3_raw_df)
ac_filtered_df = hmt.preprocess.filter_axial(ac_raw_df)

binary_mask, me3_df, ac_df = hmt.preprocess.binarize_nucleus(me3_filtered_df, ac_filtered_df, thresh=2, bin_size=50, show_plots=True)
distance_map, contour_bands = hmt.preprocess.create_radial_contours(binary_mask, show_plots=True)

step=10
print("Extracting emperical H3K27me3 distributions...")
me3_rdf, me3_adf = hmt.simulate.extract_empirical_parameters(me3_df, sdis=500, step=step)

print("Extracting emperical H3K27ac distributions...")
ac_rdf, ac_adf = hmt.simulate.extract_empirical_parameters(ac_df, sdis=500, step=step)

hmt.visualize.plot_rdf_adf(me3_rdf, me3_adf, ac_rdf, ac_adf, step=10)

In [ ]:
noise_fraction = 0.05  # fraction of domain localizations to add as uniform background noise

for spacing in [600, 1000, 1500]:
    grid_seeds = hmt.simulate.make_grid_seeds(
        n_rows=10, n_cols=10,
        spacing=spacing,
        z_values=me3_df["z [nm]"].values,
    )

    n_locs = hmt.simulate.extract_n_locs_from_rdf(me3_rdf, step=step)

    grid_sim = hmt.simulate.spawn_nanodomains(
        grid_seeds,
        rdf=me3_rdf, adf=me3_adf,
        n_locs=n_locs,
        step=step,
    )

    grid_noise = hmt.simulate.add_sim_noise(grid_sim, noise_fraction=noise_fraction)
    grid_final = pd.concat([grid_sim, grid_noise], ignore_index=True)

    print(f"Grid: {len(grid_seeds)} seeds  |  {len(grid_sim):,} domain locs  |  {len(grid_noise):,} noise locs  |  spacing = {spacing:.0f} nm")

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 5))
    hmt.visualize.plot_nanodomain_2d(grid_sim,   grid_seeds, title="No noise",                          ax=axes[0])
    hmt.visualize.plot_nanodomain_2d(grid_final, grid_seeds, title=f"With noise ({noise_fraction:.0%})", ax=axes[1])
    fig.suptitle(f"Spacing = {spacing:.0f} nm")
    plt.tight_layout()
    plt.show()